In [1]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

conn = sqlite3.connect('../data/fraud_detection.db')

print("Connected to fraud_detection.db")

Connected to fraud_detection.db


In [2]:
query1 = """
WITH fraud_by_card AS (
    SELECT 
        dc.card4 as card_network,
        COUNT(*) as total_transactions,
        SUM(ft.isFraud) as fraud_count,
        ROUND(AVG(ft.isFraud) * 100, 2) as fraud_rate_pct,
        ROUND(SUM(ft.TransactionAmt), 2) as total_amount,
        ROUND(SUM(CASE WHEN ft.isFraud = 1 THEN ft.TransactionAmt ELSE 0 END), 2) as fraud_amount
    FROM fact_transactions ft
    JOIN dim_card dc ON ft.card_id = dc.card_id
    GROUP BY dc.card4
)
SELECT 
    card_network,
    total_transactions,
    fraud_count,
    fraud_rate_pct,
    '$' || CAST(ROUND(fraud_amount, 0) AS INT) as fraud_amount_at_risk
FROM fraud_by_card
ORDER BY fraud_rate_pct DESC
"""

df_card = pd.read_sql(query1, conn)
print("Fraud summary by card network:")
print(df_card.to_string(index=False))

Fraud summary by card network:
    card_network  total_transactions  fraud_count  fraud_rate_pct fraud_amount_at_risk
        discover                6651          514            7.73              $182000
            visa              384767        13373            3.48             $1997706
      mastercard              189217         6496            3.43              $855739
american express                8328          239            2.87               $42798
         unknown                1577           41            2.60                $5601


In [3]:
query2 = """
WITH hourly_fraud AS (
    SELECT 
        hour,
        COUNT(*) as total_transactions,
        SUM(isFraud) as fraud_count,
        ROUND(AVG(isFraud) * 100, 2) as fraud_rate_pct,
        ROUND(SUM(TransactionAmt), 2) as total_amount
    FROM fact_transactions
    GROUP BY hour
),
windowed AS (
    SELECT 
        hour,
        total_transactions,
        fraud_count,
        fraud_rate_pct,
        ROUND(AVG(fraud_rate_pct) OVER (
            ORDER BY hour 
            ROWS BETWEEN 2 PRECEDING AND 2 FOLLOWING
        ), 2) as rolling_avg_fraud_rate
    FROM hourly_fraud
)
SELECT * FROM windowed
ORDER BY fraud_rate_pct DESC
LIMIT 10
"""

df_hourly = pd.read_sql(query2, conn)
print("Top 10 highest fraud rate hours (with rolling average):")
print(df_hourly.to_string(index=False))

Top 10 highest fraud rate hours (with rolling average):
 hour  total_transactions  fraud_count  fraud_rate_pct  rolling_avg_fraud_rate
    7                3704          393           10.61                    8.74
    8                2591          241            9.30                    8.40
    9                2479          223            9.00                    7.62
    6                6007          467            7.77                    7.98
    5                9701          682            7.03                    6.89
   10                3627          193            5.32                    6.11
    4               14839          770            5.19                    5.51
   11                6827          265            3.88                    4.71
    3               20802          797            3.83                    4.59
    2               26732         1002            3.75                    3.81


In [4]:
query3 = """
WITH segment_risk AS (
    SELECT 
        dc.card4 as card_network,
        dc.card6 as card_type,
        ft.hour,
        COUNT(*) as total_transactions,
        SUM(ft.isFraud) as fraud_count,
        ROUND(AVG(ft.isFraud) * 100, 2) as fraud_rate_pct,
        ROUND(SUM(CASE WHEN ft.isFraud = 1 THEN ft.TransactionAmt ELSE 0 END), 2) as amount_at_risk
    FROM fact_transactions ft
    JOIN dim_card dc ON ft.card_id = dc.card_id
    GROUP BY dc.card4, dc.card6, ft.hour
    HAVING COUNT(*) > 50
),
ranked AS (
    SELECT *,
        RANK() OVER (ORDER BY fraud_rate_pct DESC) as risk_rank
    FROM segment_risk
)
SELECT 
    risk_rank,
    card_network,
    card_type,
    hour,
    total_transactions,
    fraud_count,
    fraud_rate_pct,
    '$' || CAST(ROUND(amount_at_risk, 0) AS INT) as amount_at_risk
FROM ranked
WHERE risk_rank <= 10
ORDER BY risk_rank
"""

df_segments = pd.read_sql(query3, conn)
print("Top 10 highest risk segments:")
print(df_segments.to_string(index=False))

Top 10 highest risk segments:
 risk_rank card_network card_type  hour  total_transactions  fraud_count  fraud_rate_pct amount_at_risk
         1     discover    credit    11                  59           16           27.12          $2349
         2   mastercard    credit     8                 218           52           23.85          $5024
         3   mastercard    credit     7                 330           78           23.64          $7600
         4         visa    credit     7                 503          113           22.47         $12499
         5   mastercard    credit     9                 197           38           19.29          $8752
         6         visa    credit     9                 262           47           17.94          $6077
         7     discover    credit    12                 135           23           17.04          $8045
         8   mastercard    credit     6                 470           80           17.02          $8580
         9         visa    credit 

In [5]:
query4 = """
WITH velocity AS (
    SELECT 
        card_id,
        COUNT(*) as total_transactions,
        SUM(isFraud) as fraud_count,
        ROUND(AVG(isFraud) * 100, 2) as fraud_rate_pct,
        ROUND(AVG(TransactionAmt), 2) as avg_amount,
        ROUND(MAX(TransactionAmt), 2) as max_amount,
        ROUND(SUM(TransactionAmt), 2) as total_amount
    FROM fact_transactions
    GROUP BY card_id
    HAVING COUNT(*) >= 5
)
SELECT 
    total_transactions,
    COUNT(*) as card_count,
    ROUND(AVG(fraud_rate_pct), 2) as avg_fraud_rate,
    ROUND(AVG(avg_amount), 2) as avg_transaction_amount
FROM velocity
GROUP BY total_transactions
ORDER BY avg_fraud_rate DESC
LIMIT 10
"""

df_velocity = pd.read_sql(query4, conn)
print("Transaction velocity vs fraud rate:")
print(df_velocity.to_string(index=False))

Transaction velocity vs fraud rate:
 total_transactions  card_count  avg_fraud_rate  avg_transaction_amount
                613           1           34.91                  232.41
                162           1           33.95                   45.48
                913           1           33.41                   42.56
                890           1           28.31                   39.70
                339           1           26.55                  324.43
                748           1           24.87                   44.49
                189           2           22.49                   84.19
                972           1           21.60                   49.98
               2188           1           21.25                   40.45
                922           1           20.61                   38.89


In [6]:
# Create View 1: Fraud summary by card network
conn.execute("DROP VIEW IF EXISTS vw_fraud_summary")
conn.execute("""
CREATE VIEW vw_fraud_summary AS
SELECT 
    dc.card4 as card_network,
    dc.card6 as card_type,
    COUNT(*) as total_transactions,
    SUM(ft.isFraud) as fraud_count,
    ROUND(AVG(ft.isFraud) * 100, 2) as fraud_rate_pct,
    ROUND(SUM(CASE WHEN ft.isFraud = 1 THEN ft.TransactionAmt ELSE 0 END), 2) as amount_at_risk
FROM fact_transactions ft
JOIN dim_card dc ON ft.card_id = dc.card_id
GROUP BY dc.card4, dc.card6
""")

# Create View 2: High risk segments
conn.execute("DROP VIEW IF EXISTS vw_high_risk_segments")
conn.execute("""
CREATE VIEW vw_high_risk_segments AS
SELECT 
    dc.card4 as card_network,
    dc.card6 as card_type,
    ft.hour,
    COUNT(*) as total_transactions,
    SUM(ft.isFraud) as fraud_count,
    ROUND(AVG(ft.isFraud) * 100, 2) as fraud_rate_pct,
    ROUND(SUM(CASE WHEN ft.isFraud = 1 THEN ft.TransactionAmt ELSE 0 END), 2) as amount_at_risk
FROM fact_transactions ft
JOIN dim_card dc ON ft.card_id = dc.card_id
GROUP BY dc.card4, dc.card6, ft.hour
HAVING COUNT(*) > 50
""")

# Create View 3: Transaction velocity
conn.execute("DROP VIEW IF EXISTS vw_transaction_velocity")
conn.execute("""
CREATE VIEW vw_transaction_velocity AS
SELECT 
    card_id,
    COUNT(*) as total_transactions,
    SUM(isFraud) as fraud_count,
    ROUND(AVG(isFraud) * 100, 2) as fraud_rate_pct,
    ROUND(SUM(TransactionAmt), 2) as total_amount,
    ROUND(AVG(TransactionAmt), 2) as avg_amount
FROM fact_transactions
GROUP BY card_id
""")

conn.commit()
print("3 views created: vw_fraud_summary, vw_high_risk_segments, vw_transaction_velocity")

# Total $ at risk
at_risk = pd.read_sql("""
    SELECT 
        ROUND(SUM(CASE WHEN isFraud = 1 THEN TransactionAmt ELSE 0 END), 2) as total_fraud_amount,
        ROUND(SUM(TransactionAmt), 2) as total_amount,
        ROUND(SUM(CASE WHEN isFraud = 1 THEN TransactionAmt ELSE 0 END) / SUM(TransactionAmt) * 100, 2) as fraud_pct_of_total
    FROM fact_transactions
""", conn)
print(f"\n$ At Risk Analysis:")
print(at_risk.to_string(index=False))

3 views created: vw_fraud_summary, vw_high_risk_segments, vw_transaction_velocity

$ At Risk Analysis:
 total_fraud_amount  total_amount  fraud_pct_of_total
         3083844.86   79738948.73                3.87


In [8]:
from scipy.stats import chi2_contingency

df_chi = pd.read_sql("""
    SELECT dc.card4, ft.isFraud 
    FROM fact_transactions ft 
    JOIN dim_card dc ON ft.card_id = dc.card_id
""", conn)

contingency_table = pd.crosstab(df_chi['card4'], df_chi['isFraud'])
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("Chi-Square Test: Is fraud rate significantly different across card networks?")
print(f"  Chi-square statistic : {chi2:.2f}")
print(f"  Degrees of freedom   : {dof}")
print(f"  P-value              : {p_value:.10f}")
print()
if p_value < 0.05:
    print("  RESULT: YES — fraud rate differs significantly across card networks")
    print("  Card network is a statistically significant fraud predictor (p < 0.05)")
else:
    print("  RESULT: No significant difference found")

Chi-Square Test: Is fraud rate significantly different across card networks?
  Chi-square statistic : 368.90
  Degrees of freedom   : 4
  P-value              : 0.0000000000

  RESULT: YES — fraud rate differs significantly across card networks
  Card network is a statistically significant fraud predictor (p < 0.05)


In [9]:
conn.close()

print("=" * 55)
print("PHASE 3 SQL ANALYTICS — SUMMARY")
print("=" * 55)
print()
print("CTEs built:")
print("  - Fraud summary by card network")
print("  - High risk segments (card + type + hour)")
print("  - Transaction velocity analysis")
print("  - Rolling window fraud rate by hour")
print()
print("Window functions used:")
print("  - AVG() OVER (ROWS BETWEEN) — rolling fraud rate")
print("  - RANK() OVER (ORDER BY)    — risk segment ranking")
print()
print("3 SQL views created:")
print("  - vw_fraud_summary")
print("  - vw_high_risk_segments")
print("  - vw_transaction_velocity")
print()
print("Key business metrics:")
print("  - Total fraud amount      : $3,083,844")
print("  - Total transaction value : $79,738,948")
print("  - Fraud % of total value  : 3.87%")
print()
print("Statistical validation:")
print("  - Chi-square : 368.90")
print("  - P-value    : ~0.0000000000")
print("  - Result     : Card network is significant fraud predictor")
print()
print("Highest risk segment:")
print("  - Discover credit at hour 11 : 27.1% fraud rate")
print("  - Credit cards hours 5-9     : consistently 15-24% fraud rate")
print("=" * 55)
print("Phase 3 complete. Notebook: 03_sql_analytics.ipynb")

PHASE 3 SQL ANALYTICS — SUMMARY

CTEs built:
  - Fraud summary by card network
  - High risk segments (card + type + hour)
  - Transaction velocity analysis
  - Rolling window fraud rate by hour

Window functions used:
  - AVG() OVER (ROWS BETWEEN) — rolling fraud rate
  - RANK() OVER (ORDER BY)    — risk segment ranking

3 SQL views created:
  - vw_fraud_summary
  - vw_high_risk_segments
  - vw_transaction_velocity

Key business metrics:
  - Total fraud amount      : $3,083,844
  - Total transaction value : $79,738,948
  - Fraud % of total value  : 3.87%

Statistical validation:
  - Chi-square : 368.90
  - P-value    : ~0.0000000000
  - Result     : Card network is significant fraud predictor

Highest risk segment:
  - Discover credit at hour 11 : 27.1% fraud rate
  - Credit cards hours 5-9     : consistently 15-24% fraud rate
Phase 3 complete. Notebook: 03_sql_analytics.ipynb
